In [ ]:
dbutils.library.restartPython()
!pip install --upgrade pip
!pip install requests
!pip install pandas
%pip install openpyxl
%pip install dotenv

In [ ]:
import pandas as pd
from suporte.support_functions import *

from config import get_config
from config.secrets import get_teams_webhook_url

In [ ]:
dbutils.widgets.text("projeto", "")
dbutils.widgets.text("fonte", "api")
projeto = dbutils.widgets.get("projeto").strip()
fonte = dbutils.widgets.get("fonte").strip().lower()
cfg = get_config(projeto)  # falha alto se `projeto` estiver vazio ou nao cadastrado

if fonte not in ("api", "vsol"):
    raise ValueError(f"Widget 'fonte' invalido: '{fonte}' (valores aceitos: api, vsol)")

In [ ]:
# ------- SUMARIO DE EXECUCAO -------
execucao_steps = []

def _collect_flags(df, colunas=None):
    from pyspark.sql import functions as F
    flag_cols = colunas if colunas is not None else [c for c in df.columns if c.startswith("flag_")]
    triggered = []
    for col_name in flag_cols:
        count = df.filter(F.col(col_name).isNotNull() & (F.col(col_name) != "")).count()
        if count > 0:
            triggered.append(f"{col_name}({count})")
    return "; ".join(triggered) if triggered else "\u2014"

def persistir_log():
    from pyspark.sql import Row
    import datetime as _dt
    try:
        notebook_name = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        _execution_log_path = project_path + "/execution_log"
        _batch_ts = _dt.datetime.now().isoformat()
        _log_rows = [
            Row(
                batch_ts=_batch_ts,
                projeto=projeto,
                notebook=notebook_name.split("/")[-1],
                ordem=i,
                etapa=s["etapa"],
                status=s["status"],
                registros_lidos=int(s.get("registros_lidos") or 0),
                registros_escritos=int(s.get("registros_escritos") or 0),
                flags=str(s.get("flags") or "\u2014"),
                observacoes=str(s.get("observacoes") or ""),
                ts_registro=_dt.datetime.now().isoformat(),
            )
            for i, s in enumerate(execucao_steps)
        ]
        _log_df = spark.createDataFrame(_log_rows)
        _log_df.write.format("delta").mode("append").option("mergeSchema", "true").save(_execution_log_path)
        print(f"Log persistido: {len(_log_rows)} steps -> {_execution_log_path}")
    except Exception as _log_err:
        print(f"Aviso: falha ao persistir log de execucao \u2014 {_log_err}")

In [ ]:
project_path = "/mnt/wst/" + projeto
folder_silver_export = project_path + "/SILVER/export_continuo"

In [ ]:
# =========================
# Localizar o Excel mais recente em export_continuo
# =========================
try:
    arquivos = dbutils.fs.ls(folder_silver_export)
except Exception:
    arquivos = []

# So considera o arquivo da PROPRIA fonte -- evita que a execucao da Campo pegue e reenvie o
# Excel que a VSOL acabou de gerar no mesmo dia (ou vice-versa). Ver 03_validacoes.ipynb (sufixo
# _campo/_vsol no nome do arquivo).
sufixo_fonte = "campo" if fonte == "api" else "vsol"
excels = sorted(
    [f for f in arquivos if f.name.startswith("export_df_") and f.name.endswith(f"_{sufixo_fonte}.xlsx")],
    key=lambda f: f.name,
    reverse=True,
)

if not excels:
    execucao_steps.append({
        "etapa": "Localizacao do Excel",
        "status": "Aviso",
        "registros_lidos": 0,
        "registros_escritos": 0,
        "flags": "\u2014",
        "observacoes": f"Nenhum arquivo export_df_*.xlsx encontrado em {folder_silver_export}"
    })
    persistir_log()
    dbutils.notebook.exit("Nenhum Excel encontrado para envio")

excel_path = excels[0].path
excel_name = excels[0].name

execucao_steps.append({
    "etapa": "Localizacao do Excel",
    "status": "Sucesso",
    "registros_lidos": 1,
    "registros_escritos": 1,
    "flags": "\u2014",
    "observacoes": f"Arquivo localizado: {excel_name}"
})

print("Excel a ser enviado:", excel_path)

In [ ]:
from sharepoint_connector import upload_file
import os

# ------- ENVIO DO EXCEL PARA O SHAREPOINT -------
try:
    local_export_path = f"/tmp/{excel_name}"
    dbutils.fs.cp(excel_path, f"file:{local_export_path}")

    result = upload_file(
        folder_path=cfg["sharepoint_envio_folder"],
        local_path=local_export_path,
        filename=excel_name,
    )

    execucao_steps.append({
        "etapa": "Envio SharePoint",
        "status": "Sucesso",
        "registros_lidos": 1,
        "registros_escritos": 1,
        "flags": "\u2014",
        "observacoes": f"Enviado '{excel_name}' para {result if result else 'SharePoint'}"
    })

    os.remove(local_export_path)

except Exception as _e:
    execucao_steps.append({
        "etapa": "Envio SharePoint", "status": "Erro",
        "registros_lidos": 0, "registros_escritos": 0, "flags": "\u2014",
        "observacoes": str(_e)
    })
    persistir_log()
    raise

import pandas as pd
display(pd.DataFrame(execucao_steps))

In [ ]:
df_log_execucao = (
    spark.read.format("delta").load(project_path + "/execution_log")
    .filter(F.col("projeto") == projeto)
)

def get_ultima_etapa(nome_etapa):
    linhas = (
        df_log_execucao
        .filter(F.col("etapa") == nome_etapa)
        .orderBy(F.col("batch_ts").desc())
        .limit(1)
        .collect()
    )
    return linhas[0].asDict() if linhas else None

etapa_mapeamento     = get_ultima_etapa("Mapeamento de parametros")
etapa_amostras       = get_ultima_etapa("Amostras distintas (sample_name)")
etapa_resultados     = get_ultima_etapa("Gravacao Silver - API Validado")
etapa_stations_exec  = get_ultima_etapa("Totais de validacao - Qtd Stations Executadas")
etapa_holding        = get_ultima_etapa("Totais de validacao - Qtd Holding Time Fora do Prazo")
etapa_extra          = get_ultima_etapa("Totais de validacao - Qtd Parametros Extras")
etapa_metodo         = get_ultima_etapa("Totais de validacao - Qtd Metodos Divergentes")

mapeamento_status  = etapa_mapeamento["status"] if etapa_mapeamento else "Nao encontrado"
mapeamento_detalhe = etapa_mapeamento["observacoes"] if etapa_mapeamento else "Etapa de mapeamento nao encontrada no log"

qtd_amostras   = etapa_amostras["registros_escritos"] if etapa_amostras else 0
qtd_resultados = etapa_resultados["registros_escritos"] if etapa_resultados else 0
# --- Estacoes EXECUTADAS (com pelo menos 1 amostra recebida), nao o total planejado+executado ---
qtd_stations   = etapa_stations_exec["registros_escritos"] if etapa_stations_exec else 0

qtd_holding_fora_prazo = etapa_holding["registros_escritos"] if etapa_holding else 0
qtd_parametro_extra    = etapa_extra["registros_escritos"] if etapa_extra else 0
qtd_metodo_divergente  = etapa_metodo["registros_escritos"] if etapa_metodo else 0

# ===== Periodo de coleta processado nesta execucao (domingo a domingo, definido no notebook 01) =====
# O nome da pasta gravada na Bronze pelo notebook 01 segue o padrao:
# apiCAMPO_{inicio:yyyyMMdd}_{fim:yyyyMMdd}_{data_hora_salvamento}
output_filename = dbutils.jobs.taskValues.get(
    taskKey="fetch_from_aga_api",
    key="output_filename",
    debugValue=None
)
def parse_periodo(nome_pasta):
    try:
        _, inicio8, fim8, *_ = nome_pasta.split("_")
        inicio_fmt = f"{inicio8[6:8]}/{inicio8[4:6]}/{inicio8[0:4]}"
        fim_fmt = f"{fim8[6:8]}/{fim8[4:6]}/{fim8[0:4]}"
        return f"{inicio_fmt} a {fim_fmt}"
    except Exception:
        return "Nao disponivel"

periodo_execucao = parse_periodo(output_filename) if output_filename else "Nao disponivel"

In [ ]:
import requests
from datetime import datetime
import time

link_base = cfg.get("link_base_teams")

# ===== CARD 1 (Teams via Power Automate): Resumo executivo (visao de negocio) =====
# Sem webhook configurado pro projeto (ex.: CDM antes de criar o fluxo no Power
# Automate) o cartao e pulado -- o Excel ja foi pro SharePoint na celula anterior.
# Cadastrar o segredo depois volta a notificar, sem mexer em codigo.
webhook_resumo_secret = cfg.get("teams_webhook_resumo_secret")

if not webhook_resumo_secret:
    execucao_steps.append({
        "etapa": "Notificacao Teams - Card resumo",
        "status": "Aviso",
        "registros_lidos": 0, "registros_escritos": 0, "flags": "—",
        "observacoes": "teams_webhook_resumo_secret nao definido em config/projetos.py -- cartao resumo pulado"
    })
else:
    try:
        webhook_resumo = get_teams_webhook_url(webhook_resumo_secret)
        response_card_resumo = requests.post(
            webhook_resumo,
            json={
                "projeto": cfg["nome_exibicao"],
                "mapeamento_status": mapeamento_status,
                "mapeamento_detalhe": mapeamento_detalhe,
                "qtd_amostras": qtd_amostras,
                "qtd_resultados": qtd_resultados,
                "qtd_stations": qtd_stations,
                "periodo_execucao": periodo_execucao,
                "link_arquivo": link_base,
            }
        )
        execucao_steps.append({
            "etapa": "Notificacao Teams - Card resumo",
            "status": "Sucesso" if response_card_resumo.status_code < 300 else "Aviso",
            "registros_lidos": 1, "registros_escritos": 1, "flags": "—",
            "observacoes": f"HTTP {response_card_resumo.status_code}"
        })
    except Exception as _e:
        execucao_steps.append({
            "etapa": "Notificacao Teams - Card resumo",
            "status": "Erro",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "—",
            "observacoes": f"Falha ao enviar card resumo — confirme o segredo do webhook. Detalhe: {_e}"
        })


In [ ]:
# ===== CARD 2 (Teams via Power Automate): Resumo tecnico de validacoes =====
webhook_tecnico_secret = cfg.get("teams_webhook_tecnico_secret")

if not webhook_tecnico_secret:
    execucao_steps.append({
        "etapa": "Notificacao Teams - Card tecnico",
        "status": "Aviso",
        "registros_lidos": 0, "registros_escritos": 0, "flags": "—",
        "observacoes": "teams_webhook_tecnico_secret nao definido em config/projetos.py -- cartao tecnico pulado"
    })
else:
    try:
        webhook_tecnico = get_teams_webhook_url(webhook_tecnico_secret)
        response_card_tecnico = requests.post(
            webhook_tecnico,
            json={
                "projeto": cfg["nome_exibicao"],
                "periodo_execucao": periodo_execucao,
                "qtd_stations": qtd_stations,
                "qtd_amostras": qtd_amostras,
                "qtd_holding_fora_prazo": qtd_holding_fora_prazo,
                "qtd_parametro_extra": qtd_parametro_extra,
                "qtd_metodo_divergente": qtd_metodo_divergente,
                "link_arquivo": link_base,
            }
        )
        execucao_steps.append({
            "etapa": "Notificacao Teams - Card tecnico",
            "status": "Sucesso" if response_card_tecnico.status_code < 300 else "Aviso",
            "registros_lidos": 1,
            "registros_escritos": 1,
            "flags": "—",
            "observacoes": (
                f"HTTP {response_card_tecnico.status_code} — stations={qtd_stations}, amostras={qtd_amostras}, "
                f"holding_fora_prazo={qtd_holding_fora_prazo}, parametro_extra={qtd_parametro_extra}, "
                f"metodo_divergente={qtd_metodo_divergente}"
            )
        })
    except Exception as _e:
        execucao_steps.append({
            "etapa": "Notificacao Teams - Card tecnico",
            "status": "Erro",
            "registros_lidos": 0, "registros_escritos": 0, "flags": "—",
            "observacoes": f"Falha ao enviar card tecnico — confirme se o segredo do webhook foi configurado. Detalhe: {_e}"
        })

# ------- PERSISTIR LOG DE EXECUCAO (nb4) -------
persistir_log()
display(pd.DataFrame(execucao_steps))
